In [37]:
from scapy.all import rdpcap, UDP, Raw, IP
import math

# Calculate Shannon entropy
def shannon_entropy(data):
    if not data:
        return 0
    byte_freq = [0] * 256
    for b in data:
        byte_freq[b] += 1
    entropy = 0
    for f in byte_freq:
        if f:
            p = f / len(data)
            entropy -= p * math.log2(p)
    return entropy

# Detection thresholds
ENTROPY_THRESHOLD = 7.8
PACKET_SIZE_THRESHOLD = 1300
OPENVPN_PORT = 1194
DETECTION_PERCENT_THRESHOLD = 0.5  # percentage of suspicious packets

def detect_openvpn(pcap_path):
    packets = rdpcap(pcap_path)
    total_packets = len(packets)
    suspicious_count = 0

    for pkt in packets:
        if UDP in pkt and Raw in pkt and IP in pkt:
            udp = pkt[UDP]
            payload = bytes(pkt[Raw].load)
            entropy = shannon_entropy(payload)
            pkt_size = len(pkt)

            if (
                udp.dport == OPENVPN_PORT or udp.sport == OPENVPN_PORT
                or (entropy > ENTROPY_THRESHOLD and pkt_size > PACKET_SIZE_THRESHOLD)
            ):
                suspicious_count += 1

    detection_threshold_count = max(1, int((DETECTION_PERCENT_THRESHOLD / 100) * total_packets))

    print(f"\n[*] Total packets: {total_packets}")
    print(f"[*] Suspicious OpenVPN-like packets: {suspicious_count}")
    print(f"[*] Detection threshold (0.5%): {detection_threshold_count} packets")

    if suspicious_count >= detection_threshold_count:
        print("✅ OpenVPN traffic likely detected in the pcap file.")
    else:
        print("❌ No OpenVPN traffic detected in the pcap file.")

pcap_file = r"D:\PR\NonVPN-PCAPs-01\facebook_audio2b.pcap"
detect_openvpn(pcap_file)


[*] Total packets: 94092
[*] Suspicious OpenVPN-like packets: 3
[*] Detection threshold (0.5%): 470 packets
❌ No OpenVPN traffic detected in the pcap file.
